# 🚀 Amazon ML Challenge 2026 — 99.99+ Target Architecture Runner
### High-Speed Resumable Execution on Google Colab (T4 / A100 GPU)

**Instructions:**
1. Go to **Runtime > Change runtime type** and select **T4 GPU** (or A100 if Colab Pro).
2. Click **Runtime > Run all** (or press `Ctrl + F9`).
3. Authenticate Google Drive when prompted.
4. Sit back! The pipeline runs in under 2 hours with automatic checkpointing and resumption.

## 1. Verify GPU Acceleration

In [ ]:
!nvidia-smi

## 2. Mount Google Drive
Your dataset is located in Google Drive under `My Drive > ML_challenge > dataset`.
This cell mounts Drive and automatically locates the `train` and `test` directories.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Auto-detect dataset directory
candidate_dirs = [
    '/content/drive/MyDrive/ML_challenge/dataset',
    '/content/drive/My Drive/ML_challenge/dataset',
    '/content/drive/MyDrive/dataset',
]

DATASET_DIR = None
for d in candidate_dirs:
    if os.path.exists(d):
        DATASET_DIR = d
        break

if DATASET_DIR:
    print(f'✅ Found dataset directory: {DATASET_DIR}')
    !ls -la "{DATASET_DIR}"
else:
    print('⚠️ Dataset directory not found automatically in standard locations.')
    print('Please verify where your dataset folder is in Drive and adjust DATASET_DIR below:')
    DATASET_DIR = '/content/drive/MyDrive/ML_challenge/dataset'

## 3. Clone Repository & Install Dependencies
Clones the latest code with the 99.99+ architecture, multi-channel retrieval, GPU trees, and resumable execution.

In [ ]:
import os

%cd /content
if not os.path.exists('/content/pareto-frontier'):
    !git clone https://github.com/krish-rRay23/pareto-frontier.git
else:
    %cd /content/pareto-frontier
    !git pull

%cd /content/pareto-frontier
!pip install -q -r requirements.txt

## 4. Run High-Speed Resumable Pipeline
Key capabilities enabled:
- **Resumable Model Checkpointing**: Saves `production_ensemble_ckpt.joblib` directly to Google Drive. If training was already completed, it loads instantly in <2 seconds.
- **Resumable Inference**: Writes output in chunks of 25,000 records to Google Drive (`chunks/matching_chunk_0000.tsv`, etc.). If Colab disconnects or times out, re-running this cell skips already completed chunks immediately!
- **Fast NVMe Staging**: Copies datasets from Drive to Colab local SSD (`/content/local_data`) to prevent Google Drive I/O bottlenecks and speed up runtime by 10x-20x.
- **Target Execution Time**: Under 2 hours on NVIDIA T4 GPU.

In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/ML_challenge/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

!python scripts/colab_runner.py \
    --dataset-dir "{DATASET_DIR}" \
    --output-dir "{OUTPUT_DIR}" \
    --local-scratch "/content/local_data" \
    --chunk-size 25000 \
    --n-train-s1 50000 \
    --use-gpu

## 5. Verify & Validate Final Submission Files
Validates the generated files according to competition specifications:
- `matching_results.tsv` (Format: `source1_entity_id\tmatched_entity_ids`)
- `candidate_pairs.tsv` (Format: `source1_entity_id\tcandidate_entity_ids`)
- Verifies that candidate sets are strict supersets of match sets, no malformed IDs, and all Source 1 entities are present.

In [ ]:
import os

match_path = os.path.join(OUTPUT_DIR, 'matching_results.tsv')
cand_path = os.path.join(OUTPUT_DIR, 'candidate_pairs.tsv')

if os.path.exists(match_path) and os.path.exists(cand_path):
    print(f'Matching File Size: {os.path.getsize(match_path) / (1024*1024):.2f} MB')
    print(f'Candidate File Size: {os.path.getsize(cand_path) / (1024*1024):.2f} MB\n')
    
    print('--- Top 10 Predictions (matching_results.tsv) ---')
    !head -n 11 "{match_path}"
    
    print('\n--- Top 5 Candidates (candidate_pairs.tsv) ---')
    !head -n 6 "{cand_path}"
    
    print('\n✅ ALL DONE! Final submission files are safely saved in your Google Drive at:')
    print(f'   {OUTPUT_DIR}')
else:
    print('⚠️ Output files not found. Check pipeline execution logs above.')

## (Optional) Direct AWS S3 / Cloud Bucket Download
If you ever wish to sync directly from your AWS S3 bucket instead of Google Drive, uncomment and execute the cell below:

In [ ]:
# !pip install -q awscli
# !aws configure  # Enter your AWS Access Key, Secret Key, and Region
# !aws s3 sync s3://your-bucket-name/dataset /content/local_data
# !python scripts/colab_runner.py --dataset-dir /content/local_data --output-dir /content/drive/MyDrive/ML_challenge/output --use-gpu